# A/B-тест геймификации на образовательной платформе

**Решение: отправить механику на доработку и повторную проверку.**
Средние показатели улучшаются, но у 10–11 классов уменьшается число выполненных домашних заданий.
Кроме того, выгрузка не позволяет посчитать завершение всего курса.

В Google Colab загрузите CSV в папку `data`
или замените `DATA = Path("data")` на `DATA = Path("/content")`.

## 1. Библиотеки и настройки


In [1]:
import pandas as pd
from scipy import stats
import statsmodels.api as sm

# Эти два модуля входят в Python: пути к файлам и математические функции.
from pathlib import Path
import math

DATA = Path("data")

## 2. Загрузка и первичный осмотр

In [2]:
users = pd.read_csv(DATA / "ab_test_users_groups.csv")
events = pd.read_csv(DATA / "ab_test_events.csv")

print("Строк пользователей:", len(users))
print("Строк событий:", len(events))
display(users.head())
display(events.head())

print("Пропуски:")
display(users.isna().sum())
display(events.isna().sum())
display(events["event_type"].value_counts())

Строк пользователей: 6008
Строк событий: 179330
Пропуски:


,user_id,group,signup_date,grade
0,U300000,B,2026-03-09,5
1,U300001,B,2025-09-07,9
2,U300002,B,2026-01-11,9
3,U300003,A,2025-06-07,5
4,U300004,B,2025-12-25,11


,user_id,timestamp,event_type
0,U300281,2026-05-04 00:06:44,homework_completed
1,U300733,2026-05-04 00:19:56,homework_completed
2,U304499,2026-05-04 00:22:07,login
3,U304266,2026-05-04 00:23:42,homework_started
4,U304266,2026-05-04 00:23:42,homework_started


user_id        0
group          0
signup_date    0
grade          0
dtype: int64

user_id       0
timestamp     0
event_type    0
dtype: int64

event_type
login                 75552
homework_started      68317
homework_completed    28995
return_visit_d7        4519
return_visit_d14       1947
Name: count, dtype: int64

## 3. Что действительно можно измерить

| Показатель | Как считаем | Ограничение |
|---|---|---|
| Completion Rate курса | Рассчитать нельзя | Нет `course_completed`, идентификаторов курса/модуля и общего числа заданий |
| Выполнил хотя бы одно ДЗ | Есть хотя бы одно `homework_completed` / все пользователи группы | Это не Completion Rate курса |
| Retention D7 и D14 по меткам | Есть `return_visit_d7` или `return_visit_d14` / все пользователи группы | Используем готовые метки |
| Выполненные ДЗ на ученика | Число событий `homework_completed`, включая нули | Нет ID заданий, поэтому считаем события |
| Доля выполненных среди начатых ДЗ | Число выполнений / число стартов на ученика | Диагностика |

Основная бизнес-метрика недоступна. Четыре доступных показателя анализируем как одну семью гипотез с поправкой Холма.

In [3]:
users["signup_date"] = pd.to_datetime(users["signup_date"], errors="coerce")
events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")

print("Ошибки дат регистрации:", users["signup_date"].isna().sum())
print("Ошибки дат событий:", events["timestamp"].isna().sum())
print("Регистрация:", users["signup_date"].min(), users["signup_date"].max())
print("События:", events["timestamp"].min(), events["timestamp"].max())

known_users = events["user_id"].isin(users["user_id"])
in_period = events["timestamp"].between("2026-05-04", "2026-05-25", inclusive="left")
print("События неизвестных пользователей:", (known_users == False).sum())
print("События вне периода:", (in_period == False).sum())
display(users["group"].value_counts())
display(users["grade"].value_counts().sort_index())
display(events.groupby("event_type")["timestamp"].agg(["min", "max"]))

Ошибки дат регистрации: 0
Ошибки дат событий: 0
Регистрация: 2025-03-31 00:00:00 2026-04-04 00:00:00
События: 2026-05-04 00:06:44 2026-05-24 23:55:43
События неизвестных пользователей: 0
События вне периода: 0


group
A    3005
B    3003
Name: count, dtype: int64

grade
5     862
6     914
7     840
8     888
9     816
10    854
11    834
Name: count, dtype: int64

,min,max
event_type,,
homework_completed,2026-05-04 00:06:44,2026-05-24 23:54:44
homework_started,2026-05-04 00:23:42,2026-05-24 23:55:18
login,2026-05-04 00:22:07,2026-05-24 23:55:43
return_visit_d14,2026-05-18 18:00:00,2026-05-18 18:00:00
return_visit_d7,2026-05-11 18:00:00,2026-05-11 18:00:00


## 4. Очистка данных

Применяем одинаковые правила к A и B:

1. Удаляем точные дубликаты строк событий.
2. Исключаем пользователей, у которых указаны обе группы.
3. Отдельно рассматриваем пользователей с более чем 200 событиями за эксперимент.

У обычных пользователей максимум 52 события, у аномальных — от 711.
Это **предположительно боты или тестовые аккаунты**.

In [4]:
# Удаляем повторные строки событий.
print("Дубликаты событий:", events.duplicated().sum())
events = events.drop_duplicates()

# Находим учеников, записанных в обе группы.
groups_per_user = users.groupby("user_id")["group"].nunique()
conflict_ids = groups_per_user[groups_per_user > 1].index
print("Учеников в обеих группах:", len(conflict_ids))
display(users[users["user_id"].isin(conflict_ids)])
users_valid = users[users["user_id"].isin(conflict_ids) == False].copy()

# Находим аккаунты с более чем 200 событиями.
event_counts = events.groupby("user_id").size()
suspect_ids = event_counts[event_counts > 200].index
print("Аномально активных аккаунтов:", len(suspect_ids))
display(event_counts.sort_values(ascending=False).head(15))

users_clean = users_valid[users_valid["user_id"].isin(suspect_ids) == False].copy()
events_clean = events[events["user_id"].isin(users_clean["user_id"])].copy()
print("После очистки:", len(users_clean), "учеников и", len(events_clean), "события")

Дубликаты событий: 2
Учеников в обеих группах: 8
Аномально активных аккаунтов: 12
После очистки: 5980 учеников и 169833 события


,user_id,group,signup_date,grade
357,U300357,A,2026-03-19,9
460,U300460,B,2025-08-12,6
920,U300920,B,2025-04-09,10
1014,U301014,A,2026-03-28,10
1087,U301087,B,2025-06-27,10
1367,U301367,B,2025-09-03,8
1554,U301554,A,2025-09-25,9
5216,U305216,B,2025-07-17,9
6000,U300357,B,2026-03-19,9
6001,U300460,A,2025-08-12,6


user_id
U303729    842
U304167    829
U305378    793
U302840    790
U304266    780
U300573    771
U300733    770
U302379    765
U300281    760
U300158    738
U304499    729
U303438    711
U300628     52
U303387     51
U301601     51
dtype: int64

## 5. Sanity check: распределение по группам и SRM

Предполагаем плановое распределение 50/50.
SRM проверяем критерием хи-квадрат. H0: размеры групп соответствуют 50/50.
Для сигнала SRM используем строгий порог 0,01.

In [5]:
# При сплите 50/50 ожидаем одинаковое количество учеников в A и B.
sizes_before = users_valid["group"].value_counts().sort_index()
sizes = users_clean["group"].value_counts().sort_index()
display(sizes)
print("SRM до удаления аномалий:", stats.chisquare(sizes_before).pvalue)
print("SRM после очистки:", stats.chisquare(sizes).pvalue)

grade_balance = pd.crosstab(users_clean["grade"], users_clean["group"])
display(grade_balance)
print("p-value баланса классов:", stats.chi2_contingency(grade_balance).pvalue)

# У каждого события должен быть ровно один ученик.
logs = events_clean.merge(users_clean, on="user_id", validate="many_to_one")
print("События раньше регистрации:", (logs["timestamp"] < logs["signup_date"]).sum())

SRM до удаления аномалий: 0.9793872546704734
SRM после очистки: 0.9587469721447145
p-value баланса классов: 0.23723332342252845
События раньше регистрации: 0


group
A    2992
B    2988
Name: count, dtype: int64

group,A,B
grade,,
5,446,416
6,446,462
7,402,435
8,454,431
9,430,379
10,409,438
11,405,427


## 6. Одна строка на пользователя

Сначала считаем события каждого типа, затем присоединяем их ко **всем назначенным пользователям**.
Если у ученика нет событий, должны остаться нули.

In [6]:
# Сразу считаем таблицу с аномальными аккаунтами: она пригодится в шаге 13.
counts = pd.crosstab(events["user_id"], events["event_type"])
metrics_all = users_valid.merge(counts, on="user_id", how="left")
event_names = counts.columns
metrics_all[event_names] = metrics_all[event_names].fillna(0)

metrics_all["any_homework"] = (metrics_all["homework_completed"] > 0).astype(int)
metrics_all["retention_d7"] = (metrics_all["return_visit_d7"] > 0).astype(int)
metrics_all["retention_d14"] = (metrics_all["return_visit_d14"] > 0).astype(int)

# Нельзя делить на ноль: если ДЗ не начинали, доля неизвестна.
started = metrics_all["homework_started"].replace(0, float("nan"))
metrics_all["homework_share"] = metrics_all["homework_completed"] / started

metrics = metrics_all[metrics_all["user_id"].isin(suspect_ids) == False].copy()
metric_names = ["any_homework", "retention_d7", "retention_d14", "homework_completed"]
display(metrics.head())
display(metrics.groupby("group")[metric_names + ["homework_share"]].mean())

,user_id,group,signup_date,grade,homework_completed,homework_started,login,return_visit_d14,return_visit_d7,any_homework,retention_d7,retention_d14,homework_share
0,U300000,B,2026-03-09,5,6,11,14,1,1,1,1,1,0.545455
1,U300001,B,2025-09-07,9,3,11,14,1,1,1,1,1,0.272727
2,U300002,B,2026-01-11,9,4,11,12,1,1,1,1,1,0.363636
3,U300003,A,2025-06-07,5,3,10,10,1,1,1,1,1,0.300000
4,U300004,B,2025-12-25,11,3,12,13,0,1,1,1,0,0.250000


,any_homework,retention_d7,retention_d14,homework_completed,homework_share
group,,,,,
A,0.976270,0.726939,0.288102,3.580214,0.346066
B,0.988286,0.779786,0.362450,5.106760,0.437967


## 7. Воронка доступных действий

Ступени: логин, затем начало ДЗ, затем выполнение ДЗ. Для каждой следующей ступени ищем событие
после предыдущей, чтобы воронка соблюдала порядок. Проценты считаем от всех пользователей группы.

In [7]:
# Для каждого ученика находим первый логин.
logins = events_clean[events_clean["event_type"] == "login"]
first_login = logins.groupby("user_id")["timestamp"].min()

# Оставляем только старты ДЗ после логина.
starts = events_clean[events_clean["event_type"] == "homework_started"]
starts = starts[starts["timestamp"] >= starts["user_id"].map(first_login)]
first_start = starts.groupby("user_id")["timestamp"].min()

# Оставляем только выполнения после старта ДЗ.
completed = events_clean[events_clean["event_type"] == "homework_completed"]
completed_after_start = completed[completed["timestamp"] >= completed["user_id"].map(first_start)]
first_complete = completed_after_start.groupby("user_id")["timestamp"].min()

funnel = users_clean[["user_id", "group"]].copy()
funnel["Логин"] = funnel["user_id"].isin(first_login.index)
funnel["Начал ДЗ"] = funnel["user_id"].isin(first_start.index)
funnel["Выполнил ДЗ"] = funnel["user_id"].isin(first_complete.index)
steps = ["Логин", "Начал ДЗ", "Выполнил ДЗ"]
funnel_counts = funnel.groupby("group")[steps].sum().T
funnel_rates = funnel_counts / sizes * 100
display(funnel_counts)
display(funnel_rates.style.format("{:.1f}%").bar(vmin=0, vmax=100, color="#C9E7EB"))

group,A,B
Логин,2992,2988
Начал ДЗ,2992,2988
Выполнил ДЗ,2921,2953


group,A,B
Логин,100.0%,100.0%
Начал ДЗ,100.0%,100.0%
Выполнил ДЗ,97.6%,98.8%


## 8. Распределение выполненных домашних заданий

Показываем нули и всю обычную активность. Число ДЗ дискретное, распределение может быть асимметричным.
Для сравнения **средних** используем t-тест Уэлча: он не требует равенства дисперсий.
При почти 3 000 независимых пользователей на группу среднее оценивается достаточно устойчиво.
Дополнительно проверяем доверительный интервал бутстрэпом.

In [8]:
# Таблица-распределение: сколько учеников выполнили 0, 1, 2 и более ДЗ.
distribution = pd.crosstab(metrics["homework_completed"], metrics["group"])
display(distribution.style.bar(vmin=0, color="#C9E7EB"))

# Среднее, медиана, минимум и максимум в каждой группе.
display(metrics.groupby("group")["homework_completed"].describe())

group,A,B
homework_completed,,
0,71,35
1,265,167
2,500,284
3,686,376
4,623,452
5,449,432
6,226,367
7,105,336
8,46,232


,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
A,2992.0,3.580214,1.761692,0.0,2.0,3.0,5.0,10.0
B,2988.0,5.106760,2.545925,0.0,3.0,5.0,7.0,15.0


## 9. Гипотезы и статистические тесты

**H0:** показатель одинаков в A и B. **H1:** показатели различаются.
Ошибка I рода — объявить эффект, которого нет. Ошибка II рода — пропустить реальный эффект.

Для трёх бинарных показателей используем z-тест двух долей и 95% интервал Ньюкомба для разницы B - A.
Для среднего числа ДЗ используем t-тест Уэлча и его доверительный интервал.
Поправка Холма учитывает четыре общих сравнения. Решение о значимости принимаем по `p_holm < 0.05`.
95% интервалы относятся к каждому показателю отдельно.

`p-value` — вероятность получить такое или более сильное отклонение при верной H0.

In [9]:
# Одна небольшая функция для сравнений, которые повторяются дальше.
# a и b — значения учеников из двух групп.
def compare(a, b, binary):
    if binary:
        test = sm.stats.proportions_ztest([b.sum(), a.sum()], [len(b), len(a)])
        p_value = test[1]
        interval = sm.stats.confint_proportions_2indep(
            b.sum(), len(b), a.sum(), len(a), method="newcomb"
        )
    else:
        test = stats.ttest_ind(b, a, equal_var=False)
        p_value = test.pvalue
        interval = test.confidence_interval()
    return [a.mean(), b.mean(), b.mean() - a.mean(), interval[0], interval[1], p_value]

A = metrics[metrics["group"] == "A"]
B = metrics[metrics["group"] == "B"]
columns = ["A", "B", "diff", "ci_low", "ci_high", "p_value"]
results = pd.DataFrame(index=metric_names, columns=columns, dtype=float)

for metric in metric_names:
    binary = metric != "homework_completed"
    results.loc[metric] = compare(A[metric], B[metric], binary)

results["p_holm"] = sm.stats.multipletests(results["p_value"], method="holm")[1]
display(results)

,A,B,diff,ci_low,ci_high,p_value,p_holm
any_homework,0.976270,0.988286,0.012016,0.005361,0.018938,4.297772e-04,4.297772e-04
retention_d7,0.726939,0.779786,0.052847,0.031011,0.074620,2.134559e-06,4.269118e-06
retention_d14,0.288102,0.362450,0.074348,0.050628,0.097953,8.443589e-10,2.533077e-09
homework_completed,3.580214,5.106760,1.526546,1.415536,1.637557,3.862356e-150,1.544942e-149


**Результат:** A — 2 992 ученика, B — 2 988. Все четыре сравнения значимы после Холма.

| Метрика | A | B | Разница B − A, 95% ДИ |
|---|---:|---:|---:|
| Хотя бы одно выполненное ДЗ | 97,63% | 98,83% | +1,20 п.п. [0,54; 1,89] |
| D7 по метке | 72,69% | 77,98% | +5,28 п.п. [3,10; 7,46] |
| D14 по метке | 28,81% | 36,24% | +7,43 п.п. [5,06; 9,80] |
| Число выполненных ДЗ | 3,58 | 5,11 | +1,53 [1,42; 1,64] |

Общая воронка почти насыщена: все вошли и начали ДЗ, а хотя бы одно ДЗ выполнили более 97%.

## 10. Бутстрэп: дополнительная проверка среднего

2 000 раз случайно выбираем учеников с возвращением отдельно внутри A и B.
Каждый раз считаем разницу средних. Берём 2,5-й и 97,5-й процентили этих разниц.
Фиксируем случайное зерно, чтобы повторный запуск давал тот же результат.

In [10]:
differences = []
for i in range(2000):
    # random_state делает случайную выборку воспроизводимой.
    sample_a = A["homework_completed"].sample(frac=1, replace=True, random_state=42 + i)
    sample_b = B["homework_completed"].sample(frac=1, replace=True, random_state=2042 + i)
    differences.append(sample_b.mean() - sample_a.mean())

bootstrap_ci = pd.Series(differences).quantile([0.025, 0.975])
print("95% bootstrap-интервал B − A:")
display(bootstrap_ci)

95% bootstrap-интервал B − A:


0.025    1.419502
0.975    1.633970
dtype: float64

## 11. Мощность и MDE

Мощность 80% означает: при заданном реальном эффекте тест обнаружит его примерно в 80% повторений.
MDE — минимальный эффект, на который хватает текущего размера выборки при выбранных допущениях.

Для долей используем нормальное приближение с преобразованием Коэна h, затем возвращаем результат в п.п.
Для среднего числа ДЗ используем стандартизованную разницу и общую оценку стандартного отклонения.
Показываем обычный α = 0,05 и консервативный α = 0,05 / 4 для четырёх метрик.

In [11]:
n_a = len(A)
n_b = len(B)
mde = pd.DataFrame(index=metric_names)

for alpha in [0.05, 0.0125]:
    # Чувствительность для долей при мощности 80%.
    h = sm.stats.NormalIndPower().solve_power(
        nobs1=n_a, ratio=n_b / n_a, alpha=alpha, power=0.80
    )
    for metric in metric_names[:3]:
        baseline = A[metric].mean()
        detectable = math.sin(math.asin(math.sqrt(baseline)) + h / 2) ** 2
        mde.loc[metric, alpha] = (detectable - baseline) * 100

    # Чувствительность для среднего числа ДЗ.
    d = sm.stats.TTestIndPower().solve_power(
        nobs1=n_a, ratio=n_b / n_a, alpha=alpha, power=0.80
    )
    variance_a = A["homework_completed"].var()
    variance_b = B["homework_completed"].var()
    pooled_sd = math.sqrt(((n_a - 1) * variance_a + (n_b - 1) * variance_b) / (n_a + n_b - 2))
    mde.loc["homework_completed", alpha] = d * pooled_sd

print("Первые три строки — п.п., последняя — число ДЗ на ученика.")
display(mde.round(3))

Первые три строки — п.п., последняя — число ДЗ на ученика.


,0.0500,0.0125
any_homework,0.977,1.135
retention_d7,3.166,3.759
retention_d14,3.334,3.985
homework_completed,0.159,0.189


## 12. Различия по классам

Сначала показываем каждый класс, затем объединяем
5-7, 8-9 и 10-11 классы, чтобы упростить обсуждение с продуктовой командой.
Проверяем D7, D14 и число ДЗ в трёх сегментах и применяем Холма сразу ко всем 9 тестам.
Ниже дополнительно проверяем взаимодействие группы и сегмента для числа ДЗ.

In [12]:
segment_metrics = ["retention_d7", "retention_d14", "homework_completed"]
display(metrics.groupby(["grade", "group"])[segment_metrics].mean())

# Простое присвоение сегментов по классу.
metrics_all["segment"] = "10–11"
metrics_all.loc[metrics_all["grade"] <= 9, "segment"] = "8–9"
metrics_all.loc[metrics_all["grade"] <= 7, "segment"] = "5–7"
metrics = metrics_all[metrics_all["user_id"].isin(suspect_ids) == False].copy()

rows = []
for segment in ["5–7", "8–9", "10–11"]:
    part = metrics[metrics["segment"] == segment]
    a = part[part["group"] == "A"]
    b = part[part["group"] == "B"]
    for metric in segment_metrics:
        numbers = compare(a[metric], b[metric], metric != "homework_completed")
        rows.append([segment, metric] + numbers)

segments = pd.DataFrame(rows, columns=["segment", "metric"] + columns)
segments["p_holm_9"] = sm.stats.multipletests(segments["p_value"], method="holm")[1]
display(segments)

retention_d7  retention_d14  homework_completed
grade group                                                 
5     A          0.728700       0.284753            3.634529
      B          0.824519       0.435096            6.882212
6     A          0.755605       0.300448            3.488789
      B          0.865801       0.458874            6.742424
7     A          0.706468       0.261194            3.669154
      B          0.845977       0.439080            6.360920
8     A          0.735683       0.288546            3.533040
      B          0.774942       0.345708            4.853828
9     A          0.716279       0.293023            3.516279
      B          0.783641       0.345646            4.912929
10    A          0.713936       0.315403            3.647922
      B          0.698630       0.260274            2.990868
11    A          0.728395       0.271605            3.585185
      B          0.660422       0.245902            2.927400

,segment,metric,A,B,diff,ci_low,ci_high,p_value,p_holm_9
0,5–7,retention_d7,0.731066,0.846154,0.115087,0.083926,0.146043,5.972575e-13,2.986288e-12
1,5–7,retention_d14,0.282844,0.444783,0.161939,0.125271,0.197978,8.688897e-18,6.082228e-17
2,5–7,homework_completed,3.595054,6.660320,3.065266,2.911514,3.219018,9.182157e-261,8.263941e-260
3,8–9,retention_d7,0.726244,0.779012,0.052768,0.011587,0.093517,1.205988e-02,4.823951e-02
4,8–9,retention_d14,0.290724,0.345679,0.054955,0.010581,0.099182,1.517525e-02,4.823951e-02
5,8–9,homework_completed,3.524887,4.881481,1.356595,1.175770,1.537419,4.269395e-46,3.415516e-45
6,10–11,retention_d7,0.721130,0.679769,-0.041361,-0.084904,0.002514,6.459924e-02,1.260454e-01
7,10–11,retention_d14,0.293612,0.253179,-0.040433,-0.082998,0.002194,6.302268e-02,1.260454e-01
8,10–11,homework_completed,3.616708,2.959538,-0.657170,-0.822219,-0.492121,1.013731e-14,6.082388e-14


Дополнительная проверка. Простая линейная модель оценивает,
меняется ли эффект B между сегментами. `*` добавляет отдельные эффекты и их взаимодействия.
Устойчивые стандартные ошибки HC3 учитывают разные дисперсии. Совместный тест двух взаимодействий
проверяет H0: эффект по числу ДЗ одинаков во всех трёх сегментах.

In [13]:
# Дополнительная модель нужна для проверки, описанной выше.
model = sm.OLS.from_formula("homework_completed ~ C(group) * C(segment)", data=metrics)
model = model.fit(cov_type="HC3")
tests = model.wald_test_terms(scalar=True).table
display(tests.loc["C(group):C(segment)"])

statistic         1.046450e+03
pvalue           5.838703e-228
df_constraint     2.000000e+00
Name: C(group):C(segment), dtype: float64

## 13. Проверки устойчивости и парадокс Симпсона

**Очистка.** Повторяем основные сравнения с аномально активными пользователями.
Порог от 100 до 500 даёт тот же набор из 12 аккаунтов из-за разрыва в распределении.

**Состав групп.** Парадокс Симпсона возникает, когда различие состава групп меняет агрегированный вывод.
Чтобы проверить влияние состава, пересчитываем средние с одинаковыми весами классов в A и B.

In [14]:
# Повторяем общие сравнения, теперь включая аномальные аккаунты.
a = metrics_all[metrics_all["group"] == "A"]
b = metrics_all[metrics_all["group"] == "B"]
sensitivity = pd.DataFrame(index=metric_names, columns=columns, dtype=float)
for metric in metric_names:
    sensitivity.loc[metric] = compare(a[metric], b[metric], metric != "homework_completed")
sensitivity["p_holm"] = sm.stats.multipletests(sensitivity["p_value"], method="holm")[1]
display(sensitivity)

# Та же проверка отдельно по сегментам.
rows = []
for segment in ["5–7", "8–9", "10–11"]:
    part = metrics_all[metrics_all["segment"] == segment]
    a = part[part["group"] == "A"]
    b = part[part["group"] == "B"]
    for metric in segment_metrics:
        numbers = compare(a[metric], b[metric], metric != "homework_completed")
        rows.append([segment, metric] + numbers)
suspect_segments = pd.DataFrame(rows, columns=["segment", "metric"] + columns)
suspect_segments["p_holm_9"] = sm.stats.multipletests(suspect_segments["p_value"], method="holm")[1]
display(suspect_segments)

# Выравниваем доли классов: в обеих группах используем общие веса.
weights = metrics["grade"].value_counts(normalize=True)
standardized = pd.DataFrame(index=metric_names, columns=["A", "B", "diff"], dtype=float)
for metric in metric_names:
    means = metrics.pivot_table(index="grade", columns="group", values=metric)
    mean_a = (means["A"] * weights).sum()
    mean_b = (means["B"] * weights).sum()
    standardized.loc[metric] = [mean_a, mean_b, mean_b - mean_a]
display(standardized)

,A,B,diff,ci_low,ci_high,p_value,p_holm
any_homework,0.976310,0.988314,0.012004,0.005361,0.018913,4.242722e-04,4.242722e-04
retention_d7,0.727060,0.779633,0.052572,0.030757,0.074325,2.355311e-06,4.710622e-06
retention_d14,0.287621,0.362270,0.074649,0.050960,0.098223,6.873829e-10,2.749531e-09
homework_completed,3.968969,5.698831,1.729862,1.162400,2.297325,2.425993e-09,7.277978e-09


,segment,metric,A,B,diff,ci_low,ci_high,p_value,p_holm_9
0,5–7,retention_d7,0.731481,0.845220,0.113739,0.082597,0.144681,1.050527e-12,8.404218e-12
1,5–7,retention_d14,0.282407,0.443854,0.161447,0.124842,0.197425,9.620060e-18,8.658054e-17
2,5–7,homework_completed,3.982253,7.645675,3.663422,2.634094,4.692751,3.906724e-12,2.734707e-11
3,8–9,retention_d7,0.726554,0.779285,0.052731,0.011591,0.093440,1.203513e-02,6.017566e-02
4,8–9,retention_d14,0.290395,0.346486,0.056090,0.011733,0.100294,1.316225e-02,6.017566e-02
5,8–9,homework_completed,3.768362,5.176326,1.407964,0.636512,2.179416,3.540831e-04,2.124499e-03
6,10–11,retention_d7,0.720588,0.680139,-0.040450,-0.083963,0.003388,7.047722e-02,1.474341e-01
7,10–11,retention_d14,0.292892,0.252887,-0.040005,-0.082508,0.002558,6.546168e-02,1.474341e-01
8,10–11,homework_completed,4.165441,3.225173,-0.940268,-1.877009,-0.003527,4.914470e-02,1.474341e-01


,A,B,diff
any_homework,0.976254,0.988530,0.012277
retention_d7,0.726893,0.780262,0.053369
retention_d14,0.288031,0.362850,0.074819
homework_completed,3.581261,5.120023,1.538762


**Как читать проверку:** общие положительные результаты сохраняются с аномальными аккаунтами.
У 10-11 классов число ДЗ тоже ниже в B, но после поправки Холма снижение уже незначимо.
Значит, уверенность в величине и значимости вреда зависит от подтверждения природы этих 12 аккаунтов.
Это аргумент в пользу доработки и нового теста.
После выравнивания состава классов общие эффекты почти не меняются: смены знака из-за состава групп нет.

## 14. Что означают метки удержания

Все `return_visit_d7` записаны 11 мая в 18:00, все `return_visit_d14` — 18 мая в 18:00.
Это похоже на готовые служебные флаги относительно старта эксперимента.
Их нельзя автоматически считать D7/D14 после регистрации: ученики зарегистрированы задолго до теста.
Также это не D7/D14 после первого логина в выгрузке.

Проверим альтернативу: был ли логин ровно на 7-й или 14-й календарный день после 4 мая (старт — день 0).

In [15]:
for day in [7, 14]:
    date = pd.Timestamp("2026-05-04") + pd.Timedelta(days=day)
    visitors = logins[logins["timestamp"].dt.normalize() == date]["user_id"]
    metrics["login_day"] = metrics["user_id"].isin(visitors).astype(int)
    marker = "retention_d" + str(day)

    mismatch = (metrics["login_day"] != metrics[marker]).sum()
    print("D", day, "— несовпадений:", mismatch)
    display(metrics.groupby("group")[[marker, "login_day"]].mean())

D 7 — несовпадений: 2754
D 14 — несовпадений: 3105


,retention_d7,login_day
group,,
A,0.726939,0.546457
B,0.779786,0.594378


,retention_d14,login_day
group,,
A,0.288102,0.558489
B,0.362450,0.607430


## 15. Дополнительно: время до первого выполненного ДЗ

Считаем часы от первого логина в эксперименте до первого последующего выполнения ДЗ.
Показываем долю пользователей, у которых время удалось посчитать.
Для строгой проверки потребуется анализ времени до события с учётом цензурирования и даты назначения.

In [16]:
# Выполнение ищем после логина, даже если ученик не попал в полную воронку.
after_login = completed[completed["timestamp"] >= completed["user_id"].map(first_login)]
first_done = after_login.groupby("user_id")["timestamp"].min()
hours = (first_done - first_login).dt.total_seconds() / 3600
metrics["ttv_hours"] = metrics["user_id"].map(hours)

ttv = metrics.groupby("group")["ttv_hours"].agg(["count", "median", "mean"])
ttv["coverage_pct"] = ttv["count"] / sizes * 100
display(ttv)

,count,median,mean,coverage_pct
group,,,,
A,2921,50.283333,88.766444,97.627005
B,2953,25.233333,64.376109,98.828648


## 16. Ограничения и типичные ошибки

- **Подглядывание:** анализируем один итоговый срез. Истории промежуточных проверок нет,
  поэтому подтвердить отсутствие досрочной остановки нельзя. В новом тесте заранее фиксируем срок и размер выборки.
- **Много тестов:** используем Холма для четырёх общих показателей и отдельно для девяти сегментных.
  Альтернативное удержание и TTV служат диагностикой, по ним не принимаем отдельное решение о запуске.
- **Выбросы:** не объявляем каждый большой результат ботом. Показываем явный разрыв и проверку с включением аномалий.
- **Неравные окна:** все пользователи зарегистрированы до старта, но даты назначения не предоставлены.
  Основной анализ условен на общем старте 4 мая.
- **Независимость:** предполагаем рандомизацию по ученику. Если назначали целыми школьными классами,
  нужны идентификаторы классов и поправка на кластеризацию. `grade` — номер параллели.
- **Экономика:** данных о выручке, стоимости разработки и качестве знаний нет. AOV и окупаемость не рассчитываем.
- **Сегменты:** найденные различия требуют проверки на новой выборке. «Незначимо» не означает «эффекта нет».

## 17. Бизнес-решение

**Доработать механику и провести подтверждающий тест. На всех пользователей текущую версию не раскатывать.**

По предоставленным меткам удержание и среднее число выполненных ДЗ растут в общей выборке.
Но средний результат скрывает снижение числа выполненных ДЗ у 10–11 классов.
Для 5–7 классов есть сильный положительный сигнал: этот сегмент стоит сделать приоритетом следующего теста.
Для 8–9 классов результат слабее и зависит от показателя.

Разница удержания D14 в п.п. показывает, сколько дополнительных учеников из 100 получат метку возврата.
Например, прибавка 7,4 п.п. — примерно 74 дополнительных возврата на 1 000 пользователей при переносимости эффекта.
Это не прогноз выручки и не доказательство роста завершения курсов.

**Что нужно до запуска:**

1. Подтвердить смысл меток D7/D14 и исправить противоречивые назначения.
2. Добавить дату назначения, события завершения курса/модуля и идентификаторы заданий.
3. Пересмотреть механику для 10–11 классов и отдельно проверить её влияние на нагрузку и мотивацию.
4. Заранее зафиксировать сегменты, основную метрику, защитные показатели и минимально полезный эффект.
   Пример для обсуждения: +3 п.п. к D14. Это предложение аналитика, а не заданный бизнес-порог.
5. Рассчитать размер нового теста под этот порог и выбранную поправку на множественные сравнения.

Таким образом, полностью отказываться от идеи рано: польза для младших классов заметна.
Но исходная гипотеза о росте Completion Rate курса пока не проверена.

## 18. Сохранение результатов

Таблицы сохраняем рядом в `results`. Они нужны для сверки презентации с расчётами.
Исходные CSV остаются в папке `data`.

In [17]:
Path("results").mkdir(exist_ok=True)

results.to_csv("results/overall_results.csv")
segments.to_csv("results/segment_results.csv", index=False)
mde.to_csv("results/mde.csv")
funnel_counts.to_csv("results/funnel_counts.csv")
funnel_rates.to_csv("results/funnel_rates.csv")
sensitivity.to_csv("results/sensitivity.csv")
suspect_segments.to_csv("results/sensitivity_segments.csv", index=False)
standardized.to_csv("results/standardized.csv")
ttv.to_csv("results/ttv.csv")
print("Результаты сохранены в папке results.")

Результаты сохранены в папке results.
